In [ ]:
# ============================================
# CELL 1: Imports & Configuration
# ============================================

import requests
import pandas as pd
import time
import logging
from pathlib import Path
from typing import Dict, List, Optional
from datetime import datetime

print("Imported libraries successfully!")

In [ ]:
# ============================================
# CELL 2: Configuration Variables
# ============================================

# API URLs
CATEGORY_URL = "https://tiki.vn/api/v2/categories"
LISTINGS_URL = "https://tiki.vn/api/personalish/v1/blocks/listings"
PRODUCT_API_URL = "https://tiki.vn/api/v2/products/{}"
SHOP_API_URL = "https://api.tiki.vn/product-detail/v2/widgets/seller"
PERFORMANCE_API_URL = "https://seller-store-api.tiki.vn/ovl-performances/{}"

# Headers
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://tiki.vn/",
    "x-guest-token": "FGPAnqfVsJMDaElxYiT6z1IyUCWR3Odg"
}

# Main categories
MAIN_CATEGORIES = [
    {"id": 1703, "name": "Giày - Dép nữ"},
    {"id": 27498, "name": "Phụ kiện thời trang"},
    {"id": 44792, "name": "NGON"},
    {"id": 6000, "name": "Balo và Vali"},
    {"id": 976, "name": "Túi thời trang nữ"},
    {"id": 27616, "name": "Túi thời trang nam"}
]

# Crawl settings
DEFAULT_DELAY = 1.0
PAGES_PER_CATEGORY = 3
LIMIT_PER_PAGE = 75
REQUEST_TIMEOUT = 10

# Output settings
OUTPUT_DIR = Path("output")
CATEGORIES_FILE = "categories.csv"
PRODUCTS_FILE = "products.csv"
SHOPS_FILE = "data.csv"

print("Configuration loaded!")

In [ ]:
# ============================================
# CELL 3: Setup Logger & API Session
# ============================================

def setup_logger(name: str = "TikiCrawler") -> logging.Logger:
    """Setup logger"""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    formatter = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    
    return logger


def create_session(headers: Dict) -> requests.Session:
    """Create requests session"""
    session = requests.Session()
    session.headers.update(headers)
    return session


# Initialize
logger = setup_logger()
session = create_session(HEADERS)

print("Logger and session initialized!")

In [ ]:
# ============================================
# CELL 4: API Request Functions
# ============================================

def api_get(url: str, params: Dict = None, timeout: int = 10) -> Optional[Dict]:
    """Generic GET request"""
    try:
        response = session.get(url, params=params, timeout=timeout)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception:
        return None

print("API functions ready!")

In [ ]:
# ============================================
# CELL 5: Data Parsing Functions
# ============================================

def extract_spid(product_url: str) -> Optional[str]:
    """Extract spid from product URL"""
    try:
        if 'spid=' in product_url:
            return product_url.split('spid=')[1].split('&')[0]
    except:
        pass
    return None


def get_freeship_xtra(data: Dict) -> Optional[int]:
    """Get freeship xtra status"""
    try:
        is_free = data.get('tracking_info', {}).get('amplitude', {}).get('is_freeship_xtra')
        return 1 if is_free else 0 if is_free is not None else None
    except:
        return None


def get_authentic(data: Dict) -> Optional[int]:
    """Get authentic status"""
    try:
        is_auth = data.get('tracking_info', {}).get('amplitude', {}).get('is_authentic')
        return 1 if is_auth else 0 if is_auth is not None else None
    except:
        return None


def get_origin(data: Dict) -> Optional[str]:
    """Get product origin"""
    try:
        specs = data.get('specifications', [])
        for spec in specs:
            for attr in spec.get('attributes', []):
                if attr.get('code') == 'origin':
                    return attr.get('value')
    except:
        pass
    return None


def parse_product(data: Dict, product_url: str) -> Dict:
    """Parse product details"""
    if not data:
        return {}
    
    # Count videos
    video_count = sum(
        len(v) if isinstance(v, list) else 1
        for k, v in data.items()
        if 'video_url' in k.lower() and v
    )
    
    return {
        'price': data.get('price'),
        'original_price': data.get('original_price'),
        'discount_rate': data.get('discount_rate'),
        'quantity_sold': data.get('all_time_quantity_sold'),
        'rating_average': data.get('rating_average'),
        'review_count': data.get('review_count'),
        'is_return_policy': 1 if data.get('return_policy') else 0,
        'is_freeship_xtra': get_freeship_xtra(data),
        'is_authentic': get_authentic(data),
        'image_count': len(data.get('images', [])),
        'video_count': video_count,
        'is_brand': 1 if data.get('brand') else 0,
        'brand_name': data.get('brand', {}).get('name') if data.get('brand') else None,
        'origin': get_origin(data),
        'spid': extract_spid(product_url),
        'store_id': data.get('current_seller', {}).get('id') if data.get('current_seller') else None
    }


def parse_shop(data: Dict) -> Dict:
    """Parse shop details"""
    result = {
        'store_name': None,
        'store_review_count': None,
        'total_follower': None,
        'is_official': None
    }
    
    try:
        seller = data.get('data', {}).get('seller', {})
        if seller:
            is_official = seller.get('is_official')
            result.update({
                'store_name': seller.get('name'),
                'store_review_count': seller.get('review_count'),
                'total_follower': seller.get('total_follower'),
                'is_official': 1 if is_official else 0 if is_official is not None else None
            })
    except:
        pass
    
    return result


def parse_performance(data: Dict) -> Dict:
    """Parse shop performance"""
    if not data:
        return {}
    
    return {
        'cancel_by_seller_rate': data.get('cancel_by_seller_rate_l4w'),
        'cancel_by_seller_rate_status': data.get('cancel_by_seller_rate_l4w_status'),
        'return_rate': data.get('return_rate_l4w'),
        'return_rate_status': data.get('return_rate_l4w_status')
    }

print("Parsing functions ready!")


In [ ]:
# ============================================
# CELL 6: Crawl Categories Functions
# ============================================

def get_subcategories(parent_id: int) -> List[Dict]:
    """Get level 1 subcategories"""
    params = {"include": "children", "parent_id": parent_id}
    data = api_get(CATEGORY_URL, params)
    return data.get("data", []) if data else []


def get_products_from_category(category_id: int, category_name: str, 
                               url_key: str, root_name: str) -> List[Dict]:
    """Get products from a category"""
    products = []
    
    for page in range(1, PAGES_PER_CATEGORY + 1):
        params = {
            "limit": LIMIT_PER_PAGE,
            "sort": "top_seller",
            "page": page,
            "urlKey": url_key,
            "category": category_id
        }
        
        data = api_get(LISTINGS_URL, params)
        
        if not data:
            break
        
        page_products = data.get("data", [])
        if not page_products:
            break
        
        for prod in page_products:
            products.append({
                "product_id": prod.get("id"),
                "product_name": prod.get("name"),
                "product_url": f"https://tiki.vn/{prod.get('url_path', '')}",
                "category_id": category_id,
                "category_name": category_name,
                "category_root_name": root_name
            })
        
        logger.info(f"  Page {page}: {len(page_products)} products")
        
        paging = data.get("paging", {})
        if page >= paging.get("last_page", page):
            break
        
        time.sleep(DEFAULT_DELAY)
    
    return products


def crawl_categories(output_dir: Path) -> pd.DataFrame:
    """Crawl product listings from categories"""
    logger.info("=" * 60)
    logger.info("CRAWL CATEGORIES: Getting product listings")
    logger.info("=" * 60)
    
    all_products = []
    
    for main_cat in MAIN_CATEGORIES:
        logger.info(f"\nMain category: {main_cat['name']} (ID: {main_cat['id']})")
        
        subcategories = get_subcategories(main_cat["id"])
        logger.info(f"Found {len(subcategories)} subcategories")
        
        for i, subcat in enumerate(subcategories, 1):
            logger.info(f"\n[{i}/{len(subcategories)}] {subcat['name']}")
            
            products = get_products_from_category(
                subcat["id"], subcat["name"], 
                subcat["url_key"], main_cat["name"]
            )
            
            all_products.extend(products)
            logger.info(f"Collected {len(products)} products")
            
            time.sleep(DEFAULT_DELAY)
    
    df = pd.DataFrame(all_products)
    output_file = output_dir / CATEGORIES_FILE
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    logger.info(f"\n{'=' * 60}")
    logger.info(f"Crawl Categories Complete: {len(df)} products")
    logger.info(f"Saved to: {output_file.absolute()}")
    logger.info(f"{'=' * 60}\n")
    
    return df

print("crawl_categories() function ready!")

In [ ]:
# ============================================
# CELL 7: Crawl Products Functions
# ============================================

def crawl_products(input_file: Path, output_dir: Path) -> pd.DataFrame:
    """Crawl product details"""
    logger.info("=" * 60)
    logger.info("CRAWL PRODUCTS: Getting product details")
    logger.info("=" * 60)
    
    df = pd.read_csv(input_file)
    logger.info(f"Loaded {len(df)} products from {input_file}")
    
    # Add new columns
    new_cols = ['price', 'original_price', 'discount_rate', 'quantity_sold',
               'rating_average', 'review_count', 'is_return_policy', 
               'is_freeship_xtra', 'is_authentic', 'image_count', 'video_count',
               'is_brand', 'brand_name', 'origin', 'spid', 'store_id']
    
    for col in new_cols:
        df[col] = None
    
    # Crawl details
    success = 0
    for idx, row in df.iterrows():
        logger.info(f"[{idx + 1}/{len(df)}] Product ID: {row['product_id']}")
        
        url = PRODUCT_API_URL.format(row['product_id'])
        spid = extract_spid(row['product_url'])
        
        params = {"platform": "web", "version": "3"}
        if spid:
            params["spid"] = spid
        
        data = api_get(url, params)
        
        if data:
            parsed = parse_product(data, row['product_url'])
            for key, value in parsed.items():
                df.at[idx, key] = value
            success += 1
        
        if idx < len(df) - 1:
            time.sleep(DEFAULT_DELAY * 0.5)
    
    output_file = output_dir / PRODUCTS_FILE
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    logger.info(f"\n{'=' * 60}")
    logger.info(f"Crawl Products Complete: {success}/{len(df)} successful")
    logger.info(f"Saved to: {output_file.absolute()}")
    logger.info(f"{'=' * 60}\n")
    
    return df

print("crawl_products() function ready!")

In [ ]:
# ============================================
# CELL 8: Crawl Shops Functions
# ============================================

def crawl_shops(input_file: Path, output_dir: Path) -> pd.DataFrame:
    """Crawl shop details and performance"""
    logger.info("=" * 60)
    logger.info("CRAWL SHOPS: Getting shop details")
    logger.info("=" * 60)
    
    df = pd.read_csv(input_file)
    logger.info(f"Loaded {len(df)} products from {input_file}")
    
    # Add new columns
    new_cols = ['store_name', 'store_review_count', 'total_follower', 
               'is_official', 'cancel_by_seller_rate', 
               'cancel_by_seller_rate_status', 'return_rate', 'return_rate_status']
    
    for col in new_cols:
        df[col] = None
    
    # Cache for performance data
    performance_cache = {}
    success = 0
    
    # Crawl shop info
    for idx, row in df.iterrows():
        store_id = row['store_id']
        product_id = row['product_id']
        spid = row['spid']
        
        logger.info(f"[{idx + 1}/{len(df)}] Store ID: {store_id}")
        
        if pd.notna(store_id) and pd.notna(product_id) and pd.notna(spid):
            # Get shop details
            params = {
                "seller_id": int(store_id),
                "mpid": int(product_id),
                "spid": str(spid),
                "platform": "desktop",
                "version": "3"
            }
            
            shop_data = api_get(SHOP_API_URL, params)
            
            if shop_data:
                parsed_shop = parse_shop(shop_data)
                for key, value in parsed_shop.items():
                    df.at[idx, key] = value
                
                # Get performance (cached)
                if store_id not in performance_cache:
                    perf_url = PERFORMANCE_API_URL.format(int(store_id))
                    perf_data = api_get(perf_url)
                    
                    if perf_data:
                        performance_cache[store_id] = parse_performance(perf_data)
                    else:
                        performance_cache[store_id] = {}
                
                # Apply performance data
                perf = performance_cache[store_id]
                for key, value in perf.items():
                    df.at[idx, key] = value
                
                success += 1
        
        if idx < len(df) - 1:
            time.sleep(DEFAULT_DELAY * 2)
    
    # Remove spid column
    df = df.drop(columns=['spid'], errors='ignore')
    
    output_file = output_dir / SHOPS_FILE
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    logger.info(f"\n{'=' * 60}")
    logger.info(f"Crawl Shops Complete: {success}/{len(df)} successful")
    logger.info(f"Saved to: {output_file.absolute()}")
    logger.info(f"{'=' * 60}\n")
    
    return df

print("crawl_shops() function ready!")

In [ ]:
# ============================================
# CELL 9: Initialize Output Directory
# ============================================

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("TIKI CRAWLER INITIALIZED")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Categories to crawl: {len(MAIN_CATEGORIES)}")
print(f"Pages per category: {PAGES_PER_CATEGORY}")
print(f"Products per page: {LIMIT_PER_PAGE}")
print("=" * 60)
print("\nReady to crawl! Run the next cells to start.")

In [ ]:
# ============================================
# CELL 10: RUN - Crawl Categories
# ============================================

print("Starting: Crawl Categories...")
df_categories = crawl_categories(OUTPUT_DIR)

print(f"\nCrawl Categories Complete!")
print(f"Total products collected: {len(df_categories)}")
print(f"\nSample data:")
df_categories.head()

In [ ]:
# ============================================
# CELL 11: RUN - Crawl Products
# ============================================

print("Starting: Crawl Products...")
input_file = OUTPUT_DIR / CATEGORIES_FILE
df_products = crawl_products(input_file, OUTPUT_DIR)

print(f"\nCrawl Products Complete!")
print(f"Products with details: {len(df_products)}")
print(f"\nSample data:")
df_products.head()

In [ ]:
# ============================================
# CELL 12: RUN - Crawl Shops
# ============================================

print("Starting: Crawl Shops...")
input_file = OUTPUT_DIR / PRODUCTS_FILE
df_shops = crawl_shops(input_file, OUTPUT_DIR)

print(f"\nCrawl Shops Complete!")
print(f"Products with shop info: {len(df_shops)}")
print(f"\nSample data:")
df_shops.head()

In [ ]:
# ============================================
# CELL 13: FINAL SUMMARY & ANALYSIS
# ============================================

# Load final data
final_file = OUTPUT_DIR / SHOPS_FILE
df_final = pd.read_csv(final_file)

print("=" * 60)
print("FINAL CRAWL SUMMARY")
print("=" * 60)
print(f"Total products: {len(df_final)}")
print(f"Unique stores: {df_final['store_id'].nunique()}")
print(f"Unique categories: {df_final['category_name'].nunique()}")
print(f"Root categories: {df_final['category_root_name'].nunique()}")
print("\nPrice Statistics:")
print(f"  Average price: {df_final['price'].mean():,.0f} VND")
print(f"  Min price: {df_final['price'].min():,.0f} VND")
print(f"  Max price: {df_final['price'].max():,.0f} VND")
print(f"  Median price: {df_final['price'].median():,.0f} VND")
print("\nRating Statistics:")
print(f"  Average rating: {df_final['rating_average'].mean():.2f}")
print(f"  Total reviews: {df_final['review_count'].sum():,.0f}")
print("\nProduct Features:")
print(f"  Products with brand: {df_final['is_brand'].sum()}")
print(f"  Products with freeship: {df_final['is_freeship_xtra'].sum()}")
print(f"  Authentic products: {df_final['is_authentic'].sum()}")
print(f"  Official stores: {df_final['is_official'].sum()}")
print("\nFiles saved:")
print(f"  - {CATEGORIES_FILE}")
print(f"  - {PRODUCTS_FILE}")
print(f"  - {SHOPS_FILE}")
print(f"\nOutput directory: {OUTPUT_DIR.absolute()}")
print("=" * 60)

# Display sample
print("\nSample Final Data:")
df_final.head(10)

In [ ]:
# ============================================
# CELL 14: TOP INSIGHTS (Optional)
# ============================================

print("=" * 60)
print("TOP INSIGHTS")
print("=" * 60)

# Top categories by product count
print("\nTop 10 Categories:")
print(df_final['category_name'].value_counts().head(10))

# Top brands
print("\nTop 10 Brands:")
print(df_final['brand_name'].value_counts().head(10))

# Top stores
print("\nTop 10 Stores by Products:")
print(df_final['store_name'].value_counts().head(10))

# Price distribution by root category
print("\nAverage Price by Root Category:")
print(df_final.groupby('category_root_name')['price'].mean().sort_values(ascending=False))

print("\nAnalysis complete!")